## SargassoDB

Set up a database, just with sample information (V1V2, V4, metabolites)\
KL 5 April 2026

In [106]:
#back up and uncomment out all the cells here...useful for troubleshooting as I add new datasources

In [107]:
%reset -f
#%whos #also useful at times

In [108]:
import pandas as pd
import os
import pdb
from ftplib import FTP
from tqdm import tqdm

#need this to see the full column width
pd.set_option('display.max_colwidth', None)

In [109]:
# #now I see why Ben was deleting the database...otherwise get multiple inserts
# but I cannot get this to work as it is still in use and I am having trouble closing it.
# %tried;
# session.close()
# engine.dispose()
# def delete_db():
#     print('Deleting database')
#     db_path = 'new_database.db'
#     if os.path.exists(db_path):
# #         os.unlink(db_path)
#         os.remove(db_path)

In [110]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_testing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

# create a session factory
Session = sessionmaker(bind=engine)

# create a declarative base
Base = declarative_base()

In [111]:
# define the classes

class DiscreteInfo(Base):
    __tablename__ = 'discrete'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cruise = Column(String)
    cast = Column(String)
    niskin = Column(String)
    yyyymmdd = Column(String)
    nominalDepth = Column(String)
    V1V2data = Column(String)
    V4data = Column(String)
    mtabData = Column(String)
    
    #need this next row to get the nice output (other get a generic thing ?: <__main__.DiscreteInfo object at 0x000001A3FC7A0F70>)
    def __repr__(self):
        return f"<DiscreteInfo(bottleID='{self.bottleID}', cruise='{self.cruise}')>"


class SeqInfoV1V2(Base):
    __tablename__ = 'sequencingV1V2'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V1V2data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self):
        return f"<SeqInfoV1V2(bottleID='{self.bottleID}', filename='{self.filename}', V1V2data ='{self.V1V2data}')>"
    
class SeqInfoV4(Base):
    __tablename__ = 'sequencingV4'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    V4data = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"SeqInfoV4(id={self.id!r}, name={self.bottleID!r}, V4data={self.V4data!r})"
  
   
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"index(id={self.id!r}, filename={self.filename!r})"
    
    
class MetaboliteInfo(Base):
    __tablename__ = 'metabolites'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    dataSource = Column(String)
    
    def __repr__(self) -> str:
        return f"MetaboliteInfo(id={self.id!r}, bottleID={self.bottleID!r}, dataSource={self.dataSource!r})"
    
class MtabUntargetedInfo(Base):
    __tablename__ = 'metabolitesUntargeted'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    dataSource = Column(String)
    
    def __repr__(self) -> str:
        return f"MtabUntargetedInfo(id={self.id!r}, bottleID={self.bottleID!r}, dataSource={self.dataSource!r})"

In [112]:
# create the database tables
Base.metadata.create_all(engine)

In [113]:
# # insert some data, setup functions, one per data type
def load_discrete_info():
    print('Loading discrete sample information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'BATS_BS_COMBINED_MASTER_mini.xlsx'
    #fName = 'BATS_BS_COMBINED_MASTER_latest.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),sheet_name='DATA'))

    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = DiscreteInfo()
        db.bottleID = row['New_ID'] 
        db.cruise = row['Cruise_ID']
        db.cast = row['Cast']
        db.niskin = row['Niskin']
        db.yyyymmdd = row['yyyymmdd']
        db.nominalDepth = row['Nominal_Depth']
        session.add(db)
    
    session.commit()
    
def load_V4_sequencing_info():
    print('Loading V4 sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = SeqInfoV4()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FilenameinCyverse']
        db.V4data = fName
        session.add(db)
    
    session.commit()
    
def load_V1V2_sequencing_info():
    print('Loading V1V2 sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V1V2_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName),
                                   dtype={'Bottle ID':str,'Cruise':str,'Cast':str,'Depth':str}))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !
    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = SeqInfoV1V2()
        db.bottleID = row['BottleID'] 
        db.cast = row['Cast']
        db.filename = row['FileName']
        db.V1V2data = fName
        session.add(db)
    
    session.commit()

def load_cyverse_info():
    print('Loading sequencing information')
    dataDir = '../test_data/BIOS-SCOPE time series/'
    fName = 'files_shortList.txt'
    df = pd.read_csv(os.path.join(dataDir,fName),sep='\t',header=None,comment = '#')

    #strip off the end of the filename
    for index,row in df.iterrows():
        file = os.path.basename(row.to_string()).strip('.gz')
        df.loc[index,'filename'] = file

    session = Session()
    for index, row in tqdm(df.iterrows()):
        db = CyverseInfo()
        db.filename = row['filename'] 
        session.add(db)
    
    session.commit()

def load_metabolite_info():
    print("Loading metabolite information from MetaboLights")
    dataDir = '../test_data/'
    # start with one dataset at MetaboLights --> MTBLS2356 is Longnecker et al.
    study_id = 'MTBLS2356'
    ftp = FTP('ftp.ebi.ac.uk') #address from MetaboLights webpage
    ftp.login()
    ftpDataAddress = '/pub/databases/metabolights/studies/public/' + study_id
    ftp.cwd(ftpDataAddress)
    fileList = ftp.nlst() #can use this to make a list that will be searchable
    
    #start with the metadata about the samples 
    str = 's_' + study_id #this is the search string for the data files
    metadataFiles = [v for v in fileList if str in v] 
    metadataFiles = pd.DataFrame(metadataFiles,columns = ['files'])
    readFile = metadataFiles.loc[0,'files']

    #pdb.set_trace()
    # metadataFiles: put them here 
    # Is there a way to download an FTP file and not write it disk?
    #writeFile = 'dataDir/' + 'tempMetadata.txt'
    writeFile = os.path.join(dataDir,'tempMetadata.txt')

    with open(writeFile,'wb') as fp:
        try:
            retr_command = f"RETR {readFile}"
            ftp.retrbinary(retr_command, fp.write)
        except Exception as e: 
            print(f"Error during quit: {e}")
        except AttributeError as e: 
            print(f"AttributeError during quit: {e} - connection was likely already closed.")

    ftp.quit()
    
    # now read in the result (cannot remember why I did this in two steps)
    metadata_aboutSamples = pd.read_table(writeFile,delimiter = '\t')
    
    # pull what I can from the sample information at MetaboLights
    #for the database this is all need (is there a sample...)
    sampleNames  = metadata_aboutSamples['Source Name']

    #MetaboLights required samples to begin with a letter, I used 's' and need to strip that out 
    NewID_inMTBLS  = pd.to_numeric(sampleNames.str.strip('s')) 

    #convert the series into a dataframe:
    df = NewID_inMTBLS.reset_index() 
        
    #%run Kuj_MetabolightsData.py
    
    session = Session()
    #pdb.set_trace()
    for index,row in tqdm(df.iterrows()):
        db = MetaboliteInfo()
        db.bottleID = f"{row['Source Name']}"
        db.dataSource = study_id 
        session.add(db)
    session.commit()

In [114]:
def load_metaboliteUntargeted_info():
    print("Loading metabolite (untargeted) information from MetaboLights")
    dataDir = '../test_data/'
    # start with one dataset at MetaboLights --> MTBLS5228 is McParland et al.
    study_id = 'MTBLS5228'
    try:
        ftp = FTP('ftp.ebi.ac.uk') #address from MetaboLights webpage
        ftp.login()
        ftpDataAddress = '/pub/databases/metabolights/studies/public/' + study_id
        ftp.cwd(ftpDataAddress)
        fileList = ftp.nlst() #can use this to make a list that will be searchable
        
        #start with the metadata about the samples 
        str = 's_' + study_id #this is the search string for the data files
        metadataFiles = [v for v in fileList if str in v] 
        metadataFiles = pd.DataFrame(metadataFiles,columns = ['files'])
        readFile = metadataFiles.loc[0,'files']

        #pdb.set_trace()
        # metadataFiles: put them here 
        # Is there a way to download an FTP file and not write it disk?
        #writeFile = 'dataDir/' + 'tempMetadata.txt'
        writeFile = os.path.join(dataDir,'tempMetadata.txt')

        with open(writeFile,'wb') as fp:
            try:
                retr_command = f"RETR {readFile}"
                ftp.retrbinary(retr_command, fp.write)
            except Exception as e: 
                print(f"Error during quit: {e}")
            except AttributeError as e: 
                print(f"AttributeError during quit: {e} - connection was likely already closed.")

        ftp.quit()
        
        # now read in the result (cannot remember why I did this in two steps)
        metadata_aboutSamples = pd.read_table(writeFile,delimiter = '\t')
        
        # pull what I can from the sample information at MetaboLights
        #for the database this is all need (is there a sample...)
        sampleNames  = metadata_aboutSamples['Source Name']

        #Erin also had blanks and boooled samples, remove those and make a list (added 4/6/2026)
        #Don't really want the extra steps, but that is how I can understand this
        sampleList = [x for x in sampleNames if not x.startswith(('spool','sblank','smqblank'))]
        sampleNames = pd.Series(sampleList)
        
        #MetaboLights required samples to begin with a letter, I used 's' and need to strip that out 
        NewID_inMTBLS  = pd.to_numeric(sampleNames.str.strip('s')) 

        #convert the series into a dataframe:
        df = NewID_inMTBLS.reset_index() 
        df.columns = ['index','bottleID'] #somehow lost column labels
                   
        session = Session()
        for index,row in tqdm(df.iterrows()):
            #pdb.set_trace()
            db = MtabUntargetedInfo()
            db.bottleID = f"{row['bottleID']}" #seems like there should be a btter way to do this
            db.dataSource = study_id 
            session.add(db)
        session.commit()
    except:
        print("MetaboLights did not allow connection, dummy data OR some other error")
        session = Session()
        #pdb.set_trace()
        df = pd.DataFrame({'bottleID':['1033900707'],'dataSource':['MetaboLightsNotAvailable']})
        for index,row in tqdm(df.iterrows()):
            db = MtabUntargetedInfo()
            db.bottleID = f"{row['bottleID']}" 
            db.dataSource = row['dataSource']
            session.add(db)
        session.commit()  

In [115]:
#now run the functions
load_V4_sequencing_info()
load_V1V2_sequencing_info()
load_cyverse_info()
load_discrete_info()
load_metabolite_info()

Loading V4 sequencing information


2304it [00:00, 5142.36it/s]


Loading V1V2 sequencing information


2212it [00:00, 6017.97it/s]


Loading sequencing information


44it [00:00, 1470.65it/s]

Loading discrete sample information



2289it [00:00, 6975.96it/s]


Loading metabolite information from MetaboLights


372it [00:00, 13320.93it/s]


In [116]:
load_metaboliteUntargeted_info()

Loading metabolite (untargeted) information from MetaboLights


372it [00:00, 10889.96it/s]


In [117]:
from sqlalchemy import inspect
inspector = inspect(engine)
print(inspector.get_table_names())

['cyverse', 'discrete', 'metabolites', 'metabolitesUntargeted', 'sequencingV1V2', 'sequencingV4']


In [118]:
from sqlalchemy import create_engine, inspect, MetaData, Table
metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)

#reflect the tables so I can work on them
user_seqV4 = Table('sequencingV4', metadata_obj, autoload_with=engine)
user_seqV1V2 = Table('sequencingV1V2', metadata_obj, autoload_with=engine)
user_cy = Table('cyverse',metadata_obj,autoload_with=engine)
user_discrete = Table('discrete',metadata_obj,autoload_with=engine)
user_mtab = Table('metabolites',metadata_obj,autoload_with=engine)
user_mtabUntargeted = Table('metabolitesUntargeted',metadata_obj,autoload_with=engine)

user_mtabUntargeted

Table('metabolitesUntargeted', MetaData(), Column('id', INTEGER(), table=<metabolitesUntargeted>, primary_key=True, nullable=False), Column('bottleID', VARCHAR(), table=<metabolitesUntargeted>), Column('dataSource', VARCHAR(), table=<metabolitesUntargeted>), schema=None)

### new path from here...update the existing database
Start with the V4 data

In [119]:
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

In [120]:
#see all of what is in table
session.query(SeqInfoV4).all()

[SeqInfoV4(id=1, name='1035501701', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=2, name='1035501703', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=3, name='1035501705', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=4, name='1035501707', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=5, name='1035501709', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=6, name='1035501711', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=7, name='1035501713', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=8, name='1035501715', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=9, name='1035501717', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=10, name='1035501719', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=11, name='1035501721', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=12, name='1035501723', V4data='V4_dada2_read_info_03052026.xlsx'),
 SeqInfoV4(id=13, name='1035601501', 

In [121]:
#updating...
from sqlalchemy import update,select

# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV4.c.V4data)
    .where(user_discrete.c.bottleID == user_seqV4.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V4data=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()    

In [122]:
#move on to V1V2 (surely there is a better way to do this...)

In [123]:
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

In [124]:
# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_seqV1V2.c.V1V2data)
    .where(user_discrete.c.bottleID == user_seqV1V2.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(V1V2data=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()

In [125]:
#finally the metabolite data
# create a session factory
Session = sessionmaker(bind=engine)
session = Session()

In [126]:
# 1. Define the subquery to fetch a value from the second table
scalar_subq = (
    select(user_mtab.c.dataSource)
    .where(user_discrete.c.bottleID == user_mtab.c.bottleID)
    .limit(1)
    .scalar_subquery()
)

# 2. Use the subquery in the .values() clause of an update statement
stmt = update(user_discrete).values(mtabData=scalar_subq)

#then execute the statement
with engine.connect() as conn:
    result = conn.execute(stmt)
    conn.commit()

In [127]:
session.query(user_mtabUntargeted).all()

[(1, '1033001402', 'MTBLS5228'),
 (2, '1033001405', 'MTBLS5228'),
 (3, '1033001410', 'MTBLS5228'),
 (4, '1033001414', 'MTBLS5228'),
 (5, '1033001420', 'MTBLS5228'),
 (6, '1033001424', 'MTBLS5228'),
 (7, '1033201501', 'MTBLS5228'),
 (8, '1033201504', 'MTBLS5228'),
 (9, '1033201507', 'MTBLS5228'),
 (10, '1033201511', 'MTBLS5228'),
 (11, '1033201518', 'MTBLS5228'),
 (12, '1033201523', 'MTBLS5228'),
 (13, '1033500702', 'MTBLS5228'),
 (14, '1033500705', 'MTBLS5228'),
 (15, '1033500710', 'MTBLS5228'),
 (16, '1033500714', 'MTBLS5228'),
 (17, '1033500720', 'MTBLS5228'),
 (18, '1033500724', 'MTBLS5228'),
 (19, '1033901102', 'MTBLS5228'),
 (20, '1033901109', 'MTBLS5228'),
 (21, '1033901120', 'MTBLS5228'),
 (22, '1034101601', 'MTBLS5228'),
 (23, '1034101603', 'MTBLS5228'),
 (24, '1034101607', 'MTBLS5228'),
 (25, '1034101611', 'MTBLS5228'),
 (26, '1034101617', 'MTBLS5228'),
 (27, '1034101623', 'MTBLS5228'),
 (28, '1034301202', 'MTBLS5228'),
 (29, '1034301204', 'MTBLS5228'),
 (30, '1034301208', 'MT

In [104]:
# see if this worked
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

from sqlalchemy import Table, Column, Integer, String, MetaData

metadata = MetaData()
users = Table('discrete', metadata,
    Column('id', Integer, primary_key=True),
    Column('bottleID', String),
    Column('cruise', String),
    Column('yyyymmdd',String),
    Column('V4data',String),
    Column('V1V2data',String),
    Column('mtabData',String)
              
)

#how to execute a query
stmt = select(users)
with engine.connect() as conn:
    rows = session.execute(stmt).all()
    table_data = [row._mapping for row in rows]
    df = pd.DataFrame(table_data)
    df = df.reindex(columns = ['bottleID','cruise','yyyymmdd','V1V2data','V4data','mtabData'])

df.head()

,bottleID,cruise,yyyymmdd,V1V2data,V4data,mtabData
0,1033900707,AE1718,20170913,None,None,None
1,1033900708,AE1718,20170913,None,None,None
2,1033900709,AE1718,20170913,None,None,None
3,1033900710,AE1718,20170913,None,None,None
4,1033900711,AE1718,20170913,None,None,None


In [76]:
%run populate_db.py

Exception: File `'populate_db.py'` not found.

In [ ]:
#Stick some code below this spot as a holding zone

raise SystemExit("Stop execution here")

In [ ]:
#working on join_from ... keep this as an example

# # #now, with the discrete data, find the rows there with matching Bottle ID in the seqInfo file
# #set this up as a left outer join (all rows of discrete and only those rows of seqdata that match)
# #start tidying this up to make it useful. Plan is to ultimately send out one table with all
# #the discrete information and columns for cases where there is V1V2, V4, mtabs...

# stmt = select(
#     user_discrete.c.bottleID.label("New_ID"), 
#     user_discrete,
#     user_seqV4.c.V4data
# ).join_from(
#     user_discrete,
#     user_seqV4,
#     user_discrete.c.bottleID == user_seqV4.c.bottleID,
#     isouter=True
# )

# #session.scalars(stmt).one()
# with Session(engine) as session:
# #     for row in session.execute(stmt):
# #         print(row)
#     rows = session.execute(stmt).all()
#     table_data = [row._mapping for row in rows]
#     df = pd.DataFrame(table_data)

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

# create a SQLite database engine
#SQLALCHEMY_DATABASE_URL = "sqlite:///new_database.db"
#this will end up creating a new database everytime, but I need this for testing right now
SQLALCHEMY_DATABASE_URL = f"sqlite:///../test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
# delete_db()

#engine = create_engine(SQLALCHEMY_DATABASE_URL,echo=True)
engine = create_engine(SQLALCHEMY_DATABASE_URL)

In [ ]:
from sqlalchemy.orm import DeclarativeBase
class Base(DeclarativeBase):
    pass

In [ ]:
Base.metadata

In [ ]:
from typing import List
from typing import Optional
from sqlalchemy import Column, String
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class TestingNew(Base):
    __tablename__ = 'testingNew'
    id: Mapped[int] = mapped_column(primary_key=True)
    bottleID: Mapped[int] = mapped_column(String(10))
    cruise: Mapped[str] = mapped_column(String(10))
    cast: Mapped[int] = mapped_column(String(10))
    niskin: Mapped[int] = mapped_column(String(10))
    nominalDepth: Mapped[int] = mapped_column(String(10))
    
# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"

In [ ]:
from sqlalchemy import select,insert

select_stmt = select(
    user_discrete.c.bottleID.label("New_ID"), 
    user_discrete,
    user_seq.c.getDataHere
).join_from(
    user_discrete,
    user_seq,
    user_discrete.c.bottleID == user_seq.c.bottleID,
    isouter=True
)

insert_stmt = insert(user_discrete).from_select(
    ["id","getDataHere"],select_stmt)

# select_stmt = select(user_table.c.id, user_table.c.name + "@aol.com")
# insert_stmt = insert(address_table).from_select(
#     ["user_id", "email_address"], select_stmt
# )
print(select_stmt) #OK
print(insert_stmt) #fails: key error

In [ ]:
user_table_seqInfo.primary_key

In [ ]:
metadata_obj.create_all(engine)

In [ ]:
##now we want to declare our classes
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

# class User(Base):
#     __tablename__ = "user_account"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     name: Mapped[str] = mapped_column(String(30))
#     fullname: Mapped[Optional[str]]
#     addresses: Mapped[List["Address"]] = relationship(back_populates="user")
#     def __repr__(self) -> str:
#         return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

# class Address(Base):
#     __tablename__ = "address"
#     id: Mapped[int] = mapped_column(primary_key=True)
#     email_address: Mapped[str]
#     user_id = mapped_column(ForeignKey("user_account.id"))
#     user: Mapped[User] = relationship(back_populates="addresses")
#     def __repr__(self) -> str:
#         return f"Address(id={self.id!r}, email_address={self.email_address!r})"

#mote difference in syntax from example...this is new in SQLAlchemy 1.4

class SeqInfo(Base):
    __tablename__ = 'sequencingInfo'
    id = Column(Integer, primary_key=True, index=True)
    bottleID = Column(String)
    cast = Column(String)
    NominalDepth = Column(String)
    filename = Column(String)
    #not sure how to do this next bit yet
    #casts = relationship('Cast', back_populates='cruise')
    
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.bottleID!r}, fullname={self.filename!r})"
    
class CyverseInfo(Base):
    __tablename__ = 'cyverse'
    id = Column(Integer, primary_key=True, index=True)
    filename = Column(String)  
    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.filename!r})"

In [ ]:
Base.metadata.create_all(engine)

In [ ]:
metadata_obj

In [ ]:
#now insert data

In [ ]:
from sqlalchemy.orm import sessionmaker, Session
from datetime import datetime, time
from tqdm import tqdm

In [ ]:
def load_sequencing_info():
    print('Loading sequencing information')
    data_dir = '../test_data/BIOS-SCOPE time series/'
    fName = 'V4_dada2_read_info_03052026.xlsx'
    df = pd.DataFrame(pd.read_excel(os.path.join(data_dir,fName)))
    #strip the @#%@#^$ spaces in headers
    df.columns = df.columns.str.replace(' ','') ## actual file has a space AFTER Bottle ID !

    #df = pd.read_csv('test_data/cruise_data.csv')
    session = Session()
    for index, row in df.iterrows():
        #pdb.set_trace()
        db_si = SeqInfo()
        db_si.bottleID = row['BottleID'] 
        db_si.cast = row['Cast']
        #db_si.filename = row['FilenameInCyverse']
        session.add(db_si)
    session.commit()

In [ ]:
load_sequencing_info()

In [ ]:
#start with the sequence data, Luis gave me three lists. 
#Merge these and pull out relevant details, export to CSV file that will be read into the database
seqDir = '../test_data/Luis_fileLists'

dir_list = os.listdir(seqDir)
dir_list_full = [os.path.join(seqDir,f) for f in os.listdir(seqDir)] 